# 3D Packing Problem

#### Importing packages

In [1]:
import numpy as np 
import pandas as pd
from amplpy import AMPL

#### Data formatting

In [2]:
with pd.ExcelFile("3dpack2.xlsx") as xls:
    dfc_df = pd.read_excel(xls,"DFC")
    sku_df = pd.read_excel(xls, "SKU")

print(dfc_df.to_string())
print(sku_df.to_string())

   Type  Number of Copies  Length  Width  Height  Weight
0     1                 2       2      2       2       5
   sku  Length  Width  Height  Weight
0    1       1      1       1       1
1    2       1      1       1       1
2    3       1      1       1       1
3    4       1      1       1       1


### Notations

- $I=\{1,2,...,N\}$ is the set of SKUs, where $N$ denotes the total number of SKUs to be packed.

- $J=\{1,2,...,M\}$ is the set of DFC types, where $M$ denotes the number of different types of DFCs available for packing.

- $T_{j}$ is the set of copy indices for DFC type $j$, $\forall j \in J$.

- The $i^{th}$ SKU is characterized by its length $l_i$, width $w_i$, height $h_i$, and weight $m_i$, $\forall i \in I$.

- The $j^{th}$ DFC type is characterized by its length $L_j$, width $W_j$, height $H_j$, and maximum weight capacity $G_j$, $\forall j \in J$.

- $\mathcal{M}$ is a sufficiently large constant (big-M).

- $lr^+_{ii'jk} = 1$ if the $i^{th}$ SKU is to the left of the $i'^{th}$ SKU in copy $k$ of DFC $j$; 0 otherwise.
- $lr^-_{ii'jk} = 1$ if the $i^{th}$ SKU is to the right of the $i'^{th}$ SKU in copy $k$ of DFC $j$; 0 otherwise.
- $bf^+_{ii'jk} = 1$ if the $i^{th}$ SKU is behind the $i'^{th}$ SKU in copy $k$ of DFC $j$; 0 otherwise.
- $bf^-_{ii'jk} = 1$ if the $i^{th}$ SKU is in front of the $i'^{th}$ SKU in copy $k$ of DFC $j$; 0 otherwise.
- $bt^+_{ii'jk} = 1$ if the $i^{th}$ SKU is below the $i'^{th}$ SKU in copy $k$ of DFC $j$; 0 otherwise.
- $bt^-_{ii'jk} = 1$ if the $i^{th}$ SKU is above the $i'^{th}$ SKU in copy $k$ of DFC $j$; 0 otherwise.

- $u_{jk} \in \{0,1\}$ denotes whether copy $k$ of DFC type $j$ is used for packing.

- $s_{ijk} \in \{0,1\}$ denotes whether the $i^{th}$ SKU is packed in copy $k$ of DFC type $j$.

- $x_i,\, y_i,\, z_i \geq 0$ denote the coordinates of the left-bottom-back corner of the $i^{th}$ SKU.

- $l^a_i,\, w^a_i,\, h^a_i \in \{0,1\}$ for $a \in \{x,y,z\}$ denote whether the length, width, or height edge of the $i^{th}$ SKU is parallel to axis $a$.

---

### Formulation

The objective is to minimize the total vacant space within the assigned DFCs — equivalently, the total volume of opened DFC copies minus the fixed sum of SKU volumes.

$$
\begin{align}

\min \quad
& \sum_{j \in J}\sum_{k \in T_j} u_{jk}\,L_j W_j H_j
\;-\; \sum_{i \in I} l_i w_i h_i \tag{0} \\[6pt]

\text{s.t.} \quad
& \sum_{j \in J}\sum_{k \in T_j} s_{ijk} = 1
&& \forall\, i \in I \tag{1} \\

& u_{jk} \geq s_{ijk}
&& \forall\, i \in I,\; j \in J,\; k \in T_j \tag{2} \\[6pt]

& l_i^x + l_i^y + l_i^z = 1
&& \forall\, i \in I \tag{3} \\

& w_i^x + w_i^y + w_i^z = 1
&& \forall\, i \in I \tag{4} \\

& h_i^x + h_i^y + h_i^z = 1
&& \forall\, i \in I \tag{5} \\

& l_i^x + w_i^x + h_i^x = 1
&& \forall\, i \in I \tag{6} \\

& l_i^y + w_i^y + h_i^y = 1
&& \forall\, i \in I \tag{7} \\

& l_i^z + w_i^z + h_i^z = 1
&& \forall\, i \in I \tag{8} \\[6pt]

& x_i + l_i^x l_i + w_i^x w_i + h_i^x h_i
\;\leq\; L_j + (1 - s_{ijk})\,\mathcal{M}
&& \forall\, i \in I,\; j \in J,\; k \in T_j \tag{9} \\

& y_i + l_i^y l_i + w_i^y w_i + h_i^y h_i
\;\leq\; W_j + (1 - s_{ijk})\,\mathcal{M}
&& \forall\, i \in I,\; j \in J,\; k \in T_j \tag{10} \\

& z_i + l_i^z l_i + w_i^z w_i + h_i^z h_i
\;\leq\; H_j + (1 - s_{ijk})\,\mathcal{M}
&& \forall\, i \in I,\; j \in J,\; k \in T_j \tag{11} \\[6pt]

& \sum_{i \in I} s_{ijk}\,m_i \;\leq\; G_j
&& \forall\, j \in J,\; k \in T_j \tag{12} \\[6pt]

& x_i + l_i^x l_i + w_i^x w_i + h_i^x h_i
\;\leq\; x_{i'} + \bigl(1 - lr^+_{ii'jk} + 2(1-s_{ijk}) + 2(1-s_{i'jk})\bigr)\mathcal{M}
&& i < i',\; \forall\, i,i' \in I,\; j \in J,\; k \in T_j \tag{13} \\

& x_{i'} + l_{i'}^x l_{i'} + w_{i'}^x w_{i'} + h_{i'}^x h_{i'}
\;\leq\; x_i + \bigl(1 - lr^-_{ii'jk} + 2(1-s_{ijk}) + 2(1-s_{i'jk})\bigr)\mathcal{M}
&& i < i',\; \forall\, i,i' \in I,\; j \in J,\; k \in T_j \tag{14} \\

& y_i + l_i^y l_i + w_i^y w_i + h_i^y h_i
\;\leq\; y_{i'} + \bigl(1 - bf^+_{ii'jk} + 2(1-s_{ijk}) + 2(1-s_{i'jk})\bigr)\mathcal{M}
&& i < i',\; \forall\, i,i' \in I,\; j \in J,\; k \in T_j \tag{15} \\

& y_{i'} + l_{i'}^y l_{i'} + w_{i'}^y w_{i'} + h_{i'}^y h_{i'}
\;\leq\; y_i + \bigl(1 - bf^-_{ii'jk} + 2(1-s_{ijk}) + 2(1-s_{i'jk})\bigr)\mathcal{M}
&& i < i',\; \forall\, i,i' \in I,\; j \in J,\; k \in T_j \tag{16} \\

& z_i + l_i^z l_i + w_i^z w_i + h_i^z h_i
\;\leq\; z_{i'} + \bigl(1 - bt^+_{ii'jk} + 2(1-s_{ijk}) + 2(1-s_{i'jk})\bigr)\mathcal{M}
&& i < i',\; \forall\, i,i' \in I,\; j \in J,\; k \in T_j \tag{17} \\

& z_{i'} + l_{i'}^z l_{i'} + w_{i'}^z w_{i'} + h_{i'}^z h_{i'}
\;\leq\; z_i + \bigl(1 - bt^-_{ii'jk} + 2(1-s_{ijk}) + 2(1-s_{i'jk})\bigr)\mathcal{M}
&& i < i',\; \forall\, i,i' \in I,\; j \in J,\; k \in T_j \tag{18} \\[6pt]

& lr^+_{ii'jk} + lr^-_{ii'jk} + bf^+_{ii'jk} + bf^-_{ii'jk} + bt^+_{ii'jk} + bt^-_{ii'jk}
\;\geq\; s_{ijk} + s_{i'jk} - 1
&& i < i',\; \forall\, i,i' \in I,\; j \in J,\; k \in T_j \tag{19} \\[6pt]

& u_{j,k-1} \;\geq\; u_{jk}
&& \forall\, j \in J,\; k \in T_j : k > \min T_j \tag{20}

\end{align}
$$

#### Initialize ampl object

In [3]:
ampl = AMPL()

#### Make model

In [4]:
ampl.eval(
    """
    set SKU;             
    set DFC;              
    set Copy{DFC};                            # copies of j_th DFC 
    set axis= {'x', 'y', 'z'};
    set dim= {'Length', 'Width', 'Height'};

    param dim_sku {SKU, dim};
    param sku_Weight {SKU};
    param dim_dfc {DFC, dim};
    param dfc_Weight {DFC};
    param M;       

    var relative_position_left  {i in SKU, l in SKU, j in DFC, k in Copy[j] : i<l} binary;
    var relative_position_right {i in SKU, l in SKU, j in DFC, k in Copy[j] : i<l} binary;
    var relative_position_back  {i in SKU, l in SKU, j in DFC, k in Copy[j] : i<l} binary;
    var relative_position_front {i in SKU, l in SKU, j in DFC, k in Copy[j] : i<l} binary;
    var relative_position_below {i in SKU, l in SKU, j in DFC, k in Copy[j] : i<l} binary;
    var relative_position_above {i in SKU, l in SKU, j in DFC, k in Copy[j] : i<l} binary;

    var copy_used {j in DFC, k in Copy[j]} binary;
    var sku_in_copy {i in SKU, j in DFC, k in Copy[j]} binary;
    var sku_position {SKU, axis} >=0;

    var Length_orientation {SKU, axis} binary;
    var Width_orientation {SKU, axis} binary;
    var Height_orientation {SKU, axis} binary;

    minimize vacant_space:
    sum{j in DFC} sum{k in Copy[j]} copy_used[j,k] *dim_dfc[j,'Length']*dim_dfc[j,'Width']*dim_dfc[j,'Height'] - sum{i in SKU} dim_sku[i,'Length']*dim_sku[i,'Width']*dim_sku[i,'Height']; 

    s.t. c1 {i in SKU}: 
    sum{j in DFC} sum{k in Copy[j]} sku_in_copy[i,j,k] = 1;

    s.t. c2 {i in SKU}:
    Length_orientation[i,'x'] + Length_orientation[i,'y'] + Length_orientation[i,'z'] = 1;

    s.t. c3 {i in SKU}:
    Width_orientation[i,'x'] + Width_orientation[i,'y'] + Width_orientation[i,'z'] = 1;

    s.t. c4 {i in SKU}:
    Height_orientation[i,'x'] + Height_orientation[i,'y'] + Height_orientation[i,'z'] = 1;

    s.t. c5 {i in SKU}:
    Length_orientation[i,'x'] + Width_orientation[i,'x'] + Height_orientation[i,'x'] = 1;

    s.t. c6 {i in SKU}:
    Length_orientation[i,'y'] + Width_orientation[i,'y'] + Height_orientation[i,'y'] = 1;

    s.t. c7 {i in SKU}:
    Length_orientation[i,'z'] + Width_orientation[i,'z'] + Height_orientation[i,'z'] = 1;

    s.t. c8 {i in SKU, j in DFC, k in Copy[j]}:  
    sku_position[i,'x'] + Length_orientation[i,'x']*dim_sku[i,'Length'] + Width_orientation[i,'x']*dim_sku[i,'Width'] + Height_orientation[i,'x']*dim_sku[i,'Height'] <= dim_dfc[j, 'Length'] + (1- sku_in_copy[i,j,k])*M;
    
    s.t. c9 {i in SKU, j in DFC, k in Copy[j]}: 
    sku_position[i,'y'] + Length_orientation[i,'y']*dim_sku[i,'Length'] + Width_orientation[i,'y']*dim_sku[i,'Width'] + Height_orientation[i,'y']*dim_sku[i,'Height'] <= dim_dfc[j, 'Width'] + (1- sku_in_copy[i,j,k])*M;
    
    s.t. c10 {i in SKU, j in DFC, k in Copy[j]}: 
    sku_position[i,'z'] + Length_orientation[i,'z']*dim_sku[i,'Length'] + Width_orientation[i,'z']*dim_sku[i,'Width'] + Height_orientation[i,'z']*dim_sku[i,'Height'] <= dim_dfc[j, 'Height'] + (1- sku_in_copy[i,j,k])*M;
    
    s.t. c11 {i in SKU, j in DFC, k in Copy[j]}: 
    copy_used[j,k] >= sku_in_copy[i,j,k];

    s.t. c12 {j in DFC, k in Copy[j]}:
    sum{i in SKU} sku_in_copy[i,j,k]*sku_Weight[i] <= dfc_Weight[j];

    s.t. c13 {i in SKU, l in SKU, j in DFC, k in Copy[j] : i < l}:
    sku_position[i,'x'] + Length_orientation[i,'x']*dim_sku[i,'Length']
    + Width_orientation[i,'x']*dim_sku[i,'Width'] + Height_orientation[i,'x']*dim_sku[i,'Height']
    <= sku_position[l,'x'] + (1 - relative_position_left[i,l,j,k]) * M;

    s.t. c14 {i in SKU, l in SKU, j in DFC, k in Copy[j] : i < l}:
    sku_position[l,'x'] + Length_orientation[l,'x']*dim_sku[l,'Length']
    + Width_orientation[l,'x']*dim_sku[l,'Width'] + Height_orientation[l,'x']*dim_sku[l,'Height']
    <= sku_position[i,'x'] + (1 - relative_position_right[i,l,j,k]) * M;

    s.t. c15 {i in SKU, l in SKU, j in DFC, k in Copy[j] : i < l}:
    sku_position[i,'y'] + Length_orientation[i,'y']*dim_sku[i,'Length']
    + Width_orientation[i,'y']*dim_sku[i,'Width'] + Height_orientation[i,'y']*dim_sku[i,'Height']
    <= sku_position[l,'y'] + (1 - relative_position_back[i,l,j,k]) * M;

    s.t. c16 {i in SKU, l in SKU, j in DFC, k in Copy[j] : i < l}:
    sku_position[l,'y'] + Length_orientation[l,'y']*dim_sku[l,'Length']
    + Width_orientation[l,'y']*dim_sku[l,'Width'] + Height_orientation[l,'y']*dim_sku[l,'Height']
    <= sku_position[i,'y'] + (1 - relative_position_front[i,l,j,k]) * M;

    s.t. c17 {i in SKU, l in SKU, j in DFC, k in Copy[j] : i < l}:
    sku_position[i,'z'] + Length_orientation[i,'z']*dim_sku[i,'Length']
    + Width_orientation[i,'z']*dim_sku[i,'Width'] + Height_orientation[i,'z']*dim_sku[i,'Height']
    <= sku_position[l,'z'] + (1 - relative_position_below[i,l,j,k]) * M;

    s.t. c18 {i in SKU, l in SKU, j in DFC, k in Copy[j] : i < l}:
    sku_position[l,'z'] + Length_orientation[l,'z']*dim_sku[l,'Length']
    + Width_orientation[l,'z']*dim_sku[l,'Width'] + Height_orientation[l,'z']*dim_sku[l,'Height']
    <= sku_position[i,'z'] + (1 - relative_position_above[i,l,j,k]) * M;

    s.t. c19 {i in SKU, l in SKU, j in DFC, k in Copy[j] : i<l}:
    relative_position_left[i,l,j,k] + relative_position_right[i,l,j,k]
    + relative_position_back[i,l,j,k] + relative_position_front[i,l,j,k]
    + relative_position_below[i,l,j,k] + relative_position_above[i,l,j,k]
    >= sku_in_copy[i,j,k] + sku_in_copy[l,j,k] - 1;
    
    s.t. c20 {j in DFC, k in Copy[j] : k > 0}:
    copy_used[j,k-1] >= copy_used[j,k];
    """
    )


#### Feeding data to the model

In [5]:
ampl.set['SKU'] = sku_df['sku']
ampl.set['DFC'] = dfc_df['Type']
ampl.set['Copy'] = {
    dfc_df['Type'][i]: list(range(int(dfc_df['Number of Copies'][i])))
    for i in range(len(dfc_df))
}

ampl.param['dim_sku'] = sku_df.set_index('sku').loc[:,['Length', 'Width', 'Height']]
ampl.param['sku_Weight'] = sku_df.set_index('sku')['Weight']
ampl.param['dim_dfc'] = dfc_df.set_index('Type').loc[:,['Length', 'Width', 'Height']]
ampl.param['dfc_Weight'] = dfc_df.set_index('Type')['Weight']
ampl.param['M'] = max(
    dfc_df[['Length','Width','Height']].to_numpy().max(),
    sku_df[['Length','Width','Height']].to_numpy().max()
) * 2

#### Solve 

In [6]:
ampl.option['solver'] = 'gurobi'
ampl.option['timelimit'] = 5
ampl.solve()

Gurobi 12.0.3:Gurobi 12.0.3: optimal solution; objective 4
0 simplex iterations
1 branching node


#### Get results

In [7]:
copy_used_df = ampl.get_variable('copy_used').to_pandas()
sku_in_copy_df = ampl.get_variable('sku_in_copy').to_pandas()
sku_position_df = ampl.get_variable('sku_position').to_pandas()
Length_orientation_df = ampl.get_variable('Length_orientation').to_pandas()
Width_orientation_df = ampl.get_variable('Width_orientation').to_pandas()
Height_orientation_df = ampl.get_variable('Height_orientation').to_pandas()

status = ampl.get_value("solve_result")
print(status)

if status == "solved":
    print(ampl.get_value('vacant_space'))

solved
4


In [8]:
copy_used_df

copy_used.val
index0 index1               
1      0                   1
       1                   0

In [9]:
sku_in_copy_df

sku_in_copy.val
index0 index1 index2                 
1      1      0                     1
              1                     0
2      1      0                     1
              1                     0
3      1      0                     1
              1                     0
4      1      0                     1
              1                     0

In [10]:
sku_position_df

sku_position.val
index0 index1                  
1      x                      0
       y                      0
       z                      0
2      x                      0
       y                      1
       z                      0
3      x                      0
       y                      0
       z                      1
4      x                      0
       y                      1
       z                      1

In [11]:
Length_orientation_df

Length_orientation.val
index0 index1                        
1      x                            0
       y                            0
       z                            1
2      x                            0
       y                            0
       z                            1
3      x                            0
       y                            0
       z                            1
4      x                            0
       y                            0
       z                            1

In [12]:
Width_orientation_df

Width_orientation.val
index0 index1                       
1      x                           0
       y                           1
       z                           0
2      x                           0
       y                           1
       z                           0
3      x                           0
       y                           1
       z                           0
4      x                           0
       y                           1
       z                           0

In [13]:
Height_orientation_df

Height_orientation.val
index0 index1                        
1      x                            1
       y                            0
       z                            0
2      x                            1
       y                            0
       z                            0
3      x                            1
       y                            0
       z                            0
4      x                            1
       y                            0
       z                            0

#### Verify the results

In [14]:
print("\nAssignments:")
print(sku_in_copy_df[sku_in_copy_df['sku_in_copy.val'] == 1])

print("\nUsed bins:")
print(copy_used_df[copy_used_df['copy_used.val'] == 1])


Assignments:
                      sku_in_copy.val
index0 index1 index2                 
1      1      0                     1
2      1      0                     1
3      1      0                     1
4      1      0                     1

Used bins:
               copy_used.val
index0 index1               
1      0                   1
